In [ ]:
# daum 사이트에서 뉴스정보를 읽어 텍스트 파일로 저장 후 유사도 확인
# 형태소 분석, Word2Vec, 유사도 분석 ...

!pip install konlpy
!pip install gensim
!pip install koreanize_matplotlib

In [ ]:
import pandas as pd
from konlpy.tag import Okt
okt = Okt()

with open('daumnews.txt', 'r', encoding='utf8') as f:
    lines = f.read().splitlines()

# print(lines)

word_freq = {}    # 명사만 추출해 기억
for line in lines:
    nouns = [word for word, tag in okt.pos(line) if tag == 'Noun' and len(word) > 1]
    # print(nouns)
    for noun in nouns:
        word_freq[noun] = word_freq.get(noun, 0) + 1

print(word_freq)

# 단어 건수별 내림차순 정렬해 DataFrame에 저장
sortData = sorted(word_freq.items(), key=lambda dul:(-dul[1], dul[0]))
print(sortData)
df = pd.DataFrame(sortData, columns=['단어', '빈도수'])
print(df.head(10))  # 180 rows x 2 columns

print()
# df -> csv 파일로 저장
df.to_csv('nlp2word.csv', index=False, encoding='utf-8-sig')

df = pd.read_csv('nlp2word.csv', encoding='utf-8-sig')
print(df.head(3))

In [ ]:
# 유사도 확인 ---------
# 원본 파일에서 명사, 동사 추출
with open('nlp2word_freq.txt', 'w', encoding='utf-8') as fi:
    for line in lines:
        tokens = okt.pos(line, stem=True)
        words = [word for word, tag in tokens if tag in ['Noun', 'Verb'] and len(word) > 1]
        if words:
            fi.write(' '.join(words) + '\n')

from gensim.models import word2vec

# LineSentence : 텍스트를 한 줄씩 읽어 단어 리스트로 변환
sentences = word2vec.LineSentence('nlp2word_freq.txt')
print(sentences)

model = word2vec.Word2Vec(sentences=sentences, vector_size=100, window=10, min_count=1, sg=1)
print(model)

# 학습된 모델 저장
model.save('nlp2model.model')

# 저장된 모델 읽기
model = word2vec.Word2Vec.load('nlp2model.model')
print(model.wv.index_to_key[:5])
print('은행' in model.wv.key_to_index)
print('파이썬' in model.wv.key_to_index)

print('양자컴퓨터 유사한 단어 출력')
print(model.wv.most_similar('양자컴퓨터'))

print()
# 두 단어의 벡터를 더한 결과에 가장 가까운 단어
print(model.wv.most_similar(positive=['양자컴퓨터','은행']))
print(model.wv.most_similar(positive=['양자컴퓨터','은행'], negative=['기술']))


In [ ]:
# 시각화 - 유사도 기반 단어간 관계
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.decomposition import PCA
import platform  # 환경정보 확인용

target_word = '양자컴퓨터'

similar_words = model.wv.most_similar(target_word, topn=10)
print('similar_words : ', similar_words)

# 단어 리스트 작성
words = [target_word] + [word for word, _ in similar_words]
print('words : ', words)  # ['양자컴퓨터', '거래소', '상황', '만약',...

# 단어 벡터 추출
word_vectors = [model.wv[word] for word in words]
# print('word_vectors : ', word_vectors[0])

# 차원 축소
pca = PCA(n_components=2)
points = pca.fit_transform(word_vectors)
print('points : ', points[0])

plt.figure(figsize=(6, 5))
for i, word in enumerate(words):
    x, y = points[i]
    plt.scatter(x, y, color='blue' if i == 0 else 'black')
    plt.text(x, y, word, fontsize=10, color='red' if i == 0 else 'black')

plt.title(f'Word2Vec 유사 단어 시각화 (기준단어:{target_word})')
plt.grid(True)
plt.show()

In [ ]:
# 단어들을 의미적으로 군집화
from sklearn.cluster import KMeans
# print(model.wv.key_to_index)
filtered_words = [word for word in words if word in model.wv.key_to_index]
print(filtered_words)
vectors = [model.wv[word] for word in filtered_words]
# print(vectors)

# KMeans 클러스터링
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(vectors)

# PCA로 차원 축소
pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(vectors)
centers = pca.transform(kmeans.cluster_centers_)

colors = ['red', 'blue', 'green', 'orange', 'purple']

plt.figure(figsize=(10, 7))

for i, word in enumerate(filtered_words):
    x, y = reduced_vectors[i]
    plt.scatter(x, y, color=colors[labels[i]], s=120, edgecolor='black')
    plt.text(x + 0.005, y + 0.005, word, fontsize=10)

# 클러스터 중심점 표현
for i, (cx, cy) in enumerate(centers):
    plt.scatter(cx, cy, color=colors[i], s=200, marker='X', edgecolor='black', label=f'Cluster{i + 1}')

plt.title('Word2Vec 단어 의미 군집화')
plt.legend(title='군집')
plt.grid(True)
plt.tight_layout()
plt.show()

# 군집별 단어 리스트 출력
from collections import defaultdict

cluster_dict = defaultdict(list)
for word, label in zip(filtered_words, labels):
    cluster_dict[label].append(word)

print(cluster_dict)
for cid, word_list in cluster_dict.items():
    print(f'Cluser {cid + 1} 번째 군집 소속 단어 : {', '.join(word_list)}')


In [ ]:
# 계층적 군집분석 - 덴드로그램
from scipy.cluster.hierarchy import dendrogram, linkage
import numpy as np
vectors = np.array([model.wv[word] for word in filtered_words])

linkage_matrix = linkage(vectors, method='ward')

plt.figure(figsize=(10,5))
dendrogram(linkage_matrix, labels=filtered_words, leaf_font_size=10)
plt.title('Word2Vec으로 비계층적 군집')
plt.xlabel('단어')
plt.ylabel('거리')
plt.tight_layout()
plt.show()